# Analyse des recherches sur les accords

In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
import pandas as pd
from analysis.connectors.matomo import MatomoSQLConnector

matomo = MatomoSQLConnector()
await matomo.connect()

In [ ]:
# - 10 juin / now
interval_start = '2026-07-11 00:00:00'
interval_stop = '2026-08-31 00:00:00'

In [ ]:
columns = ['idvisit', 'action_url']

range_query = f"""
        SELECT {", ".join(columns)} FROM matomo_partitioned
        WHERE action_timestamp >= '{interval_start}'
          AND action_timestamp < '{interval_stop}'
          AND action_eventaction = 'show_accords' 
          AND action_eventname = ''
        """

In [ ]:
query_visits =  range_query + f"""
          ORDER BY action_timestamp asc;
    """

visits_data = await matomo.run_query(query_visits)

In [ ]:
visits_df = pd.DataFrame(visits_data, columns=columns)

In [ ]:
visits_df

## Récupération des SIRET recherchés

Dans une URL, on a le SIRET disponible, on l'extrait afin d'afin celui utilisé pour récupérer les accords : https://code.travail.gouv.fr/outils/convention-collective/entreprise/80136322701678?q=toto

In [ ]:
import re
import json
import base64
from urllib.parse import urlparse, parse_qs

SIRET_RE = re.compile(r"/outils/convention-collective/entreprise/([^/?#]+)")


def parse_entreprise_url(url: str) -> dict:
    """Extrait le siret, le paramètre q et le paramètre cp (brut) d'une URL entreprise."""
    if not isinstance(url, str):
        return {"siret": None, "q": None, "cp": None}

    parsed = urlparse(url)
    m = SIRET_RE.search(parsed.path)
    # parse_qs tolère les espaces non encodés (ex: ?q=Gard diffusion)
    qs = parse_qs(parsed.query, keep_blank_values=True)

    return {
        "siret": m.group(1) if m else None,
        "q": qs.get("q", [None])[0],
        "cp": qs.get("cp", [None])[0],
    }

In [ ]:
urls = visits_df["action_url"].dropna().drop_duplicates()
parsed_df = pd.DataFrame(
    [parse_entreprise_url(u) for u in urls],
    index=urls.values,
)

visits_df = visits_df.join(parsed_df, on="action_url")

In [ ]:
visits_df

In [ ]:
FLAG_SANS = "Sans q ni cp"
FLAG_Q = "q seul"
FLAG_CP = "cp seul"
FLAG_Q_CP = "q + cp"

# L'ordre pilote aussi l'ordre des parts du camembert
FLAG_ORDER = [FLAG_SANS, FLAG_Q, FLAG_CP, FLAG_Q_CP]


def _filled(v) -> bool:
    return isinstance(v, str) and v.strip() != ""


def build_flag(row) -> str:
    has_q, has_cp = _filled(row["q"]), _filled(row["cp"])
    # ajouter ici les cas suivants (ex: q == siret, cp décodé invalide, ...)
    if has_q and has_cp:
        return FLAG_Q_CP
    if has_q:
        return FLAG_Q
    if has_cp:
        return FLAG_CP
    return FLAG_SANS


visits_df["url_flag"] = visits_df.apply(build_flag, axis=1)
visits_df["url_flag"] = pd.Categorical(visits_df["url_flag"], categories=FLAG_ORDER, ordered=True)

In [ ]:
import matplotlib.pyplot as plt

# Un slot de couleur par flag : ajoute l'entrée du prochain cas ici.
FLAG_COLORS = {
    FLAG_SANS: "#2a78d6",
    FLAG_Q: "#eb6834",
    FLAG_CP: "#1baf7a",
    FLAG_Q_CP: "#eda100",
}

scope = visits_df[visits_df["siret"].notna()]
counts = scope["url_flag"].value_counts().reindex(FLAG_ORDER).fillna(0).astype(int)
counts = counts[counts > 0]  # on n'affiche pas les cas absents

total = int(counts.sum())

fig, ax = plt.subplots(figsize=(6.5, 5))
wedges, _, autotexts = ax.pie(
    counts.values,
    labels=[f"{flag}\n(n={n:,})".replace(",", " ") for flag, n in counts.items()],
    colors=[FLAG_COLORS[f] for f in counts.index],
    autopct=lambda p: f"{p:.1f}%",
    startangle=90,
    counterclock=False,
    pctdistance=0.78,
    wedgeprops={"width": 0.42, "edgecolor": "#fcfcfb", "linewidth": 2},
    textprops={"color": "#52514e", "fontsize": 10},
)
for t in autotexts:
    t.set_color("#0b0b0b")
    t.set_fontweight("bold")

ax.set_title(
    f"Paramètres présents sur les URLs entreprise\n{total:,} actions".replace(",", " "),
    color="#0b0b0b",
    fontsize=12,
    pad=16,
)
ax.axis("equal")
plt.tight_layout()
plt.show()

# Le détail chiffré, utile pour vérifier ce que montre le graphe
print((counts / total * 100).round(2).to_frame("pct").assign(n=counts))

## Analyse des SIRET

In [ ]:
from pathlib import Path
import duckdb, pandas as pd, requests

DATA_DIR  = Path("./data_sirene")   # cache du fichier Sirene (~2,2 Go)
SIRET_COL = "siret"                 # colonne SIRET dans visits_df
COMPTER   = "actifs"                # "actifs" ou "tous" : quels établissements
                                    # secondaires comptent pour le flag
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
DATASET = "base-sirene-des-entreprises-et-de-leurs-etablissements-siren-siret"


def telecharger_sirene(motif="stocketablissement") -> Path:
    fichier = f"stock-{motif}-parquet.parquet"
    meta = requests.get(f"https://www.data.gouv.fr/api/1/datasets/{DATASET}/", timeout=60).json()
    cands = [r for r in meta["resources"] if (r.get("url") or "").endswith(fichier)]
    if not cands:
        raise RuntimeError(f"{fichier} introuvable sur data.gouv.fr")
    url = sorted(cands, key=lambda r: r.get("last_modified") or "")[-1]["url"]

    cible = DATA_DIR / f"{motif}_{url.rsplit('/', 2)[-2]}.parquet"
    if cible.exists():
        print(f"Déjà en cache : {cible.name} ({cible.stat().st_size/1e9:.2f} Go)")
        return cible

    tmp = cible.with_suffix(".part")
    deja = tmp.stat().st_size if tmp.exists() else 0
    entetes = {"Range": f"bytes={deja}-"} if deja else {}
    with requests.get(url, stream=True, headers=entetes, timeout=(30, 300)) as r:
        r.raise_for_status()
        total = deja + int(r.headers.get("Content-Length", 0))
        with open(tmp, "ab") as f:
            for chunk in r.iter_content(1 << 22):
                f.write(chunk); deja += len(chunk)
                print(f"\r{deja/1e9:6.2f} / {total/1e9:.2f} Go", end="")
    tmp.rename(cible)
    print(f"\nTéléchargé : {cible.name}")
    return cible


PARQUET = telecharger_sirene()

In [ ]:
def normaliser_siret(serie: pd.Series) -> pd.Series:
    s = serie.astype("string").fillna("").str.replace(r"\D", "", regex=True)
    s = s.where(s.str.len() != 13, "0" + s)      # zéro de tête perdu par Excel / BDD
    return s.where(s.str.fullmatch(r"\d{14}"))   # <NA> si le format est invalide


visits_df["siret_norm"] = normaliser_siret(visits_df[SIRET_COL])

cles = (visits_df.loc[visits_df["siret_norm"].notna(), "siret_norm"]
                 .drop_duplicates().to_frame("siret"))
cles["siren"] = cles["siret"].str[:9]

print(f"{len(visits_df):,} lignes · {len(cles):,} SIRET distincts valides · "
      f"{int(visits_df['siret_norm'].isna().sum()):,} au format invalide".replace(",", " "))

In [ ]:
REQUETE = """
WITH etabs AS MATERIALIZED (
    SELECT siren, siret,
           lower(CAST(etablissementSiege AS VARCHAR)) = 'true' AS est_siege,
           etatAdministratifEtablissement = 'A'                AS actif
    FROM read_parquet($parquet)
    WHERE siren IN (SELECT siren FROM cles)
),
par_siren AS (
    SELECT siren,
           max(siret) FILTER (est_siege)               AS siret_siege,
           count(*)   FILTER (NOT est_siege)           AS nb_sec_tous,
           count(*)   FILTER (NOT est_siege AND actif) AS nb_sec_actifs
    FROM etabs GROUP BY siren
)
SELECT c.siret,
       c.siren,
       e.siret IS NOT NULL          AS existe,
       coalesce(e.est_siege, FALSE) AS est_siege,
       e.actif                      AS etablissement_actif,
       p.siret_siege,
       coalesce(p.nb_sec_tous, 0)   AS nb_sec_tous,
       coalesce(p.nb_sec_actifs, 0) AS nb_sec_actifs
FROM cles c
LEFT JOIN etabs     e ON e.siret = c.siret
LEFT JOIN par_siren p ON p.siren = c.siren
"""

con = duckdb.connect()
con.register("cles", cles)
lookup = con.execute(REQUETE, {"parquet": str(PARQUET)}).fetchdf()
con.close()

nb_sec = lookup["nb_sec_actifs"] if COMPTER == "actifs" else lookup["nb_sec_tous"]

lookup["flag"] = "non résolu"
lookup.loc[lookup["existe"] & lookup["est_siege"] & (nb_sec > 0),  "flag"] = "siège social (avec établissement)"
lookup.loc[lookup["existe"] & lookup["est_siege"] & (nb_sec == 0), "flag"] = "siège social (sans établissement)"
lookup.loc[lookup["existe"] & ~lookup["est_siege"],                "flag"] = "établissement"

lookup["siret_siege"] = lookup["siret_siege"].where(lookup["existe"])
lookup.loc[~lookup["existe"], ["nb_sec_tous", "nb_sec_actifs"]] = pd.NA

visits_df = visits_df.merge(
    lookup[["siret", "flag", "siret_siege", "nb_sec_actifs", "etablissement_actif"]]
          .rename(columns={"siret": "siret_norm", "nb_sec_actifs": "nb_etab_secondaires_actifs"}),
    on="siret_norm", how="left",
)
visits_df["flag"] = visits_df["flag"].fillna("non résolu")

visits_df[[SIRET_COL, "flag", "siret_siege", "nb_etab_secondaires_actifs"]].head(10)

In [ ]:
ORDRE = ["siège social (avec établissement)",
         "siège social (sans établissement)",
         "établissement",
         "non résolu"]

n = len(visits_df)
compte = visits_df["flag"].value_counts()
resume = pd.DataFrame({"flag": ORDRE, "n": [int(compte.get(f, 0)) for f in ORDRE]})
resume["% du total"] = (100 * resume["n"] / n).round(2)

n_res = int(resume.loc[resume["flag"] != "non résolu", "n"].sum())
resume["% des résolus"] = [round(100 * v / n_res, 2) if n_res and f != "non résolu" else None
                           for f, v in zip(resume["flag"], resume["n"])]

n_siege = int(compte.get(ORDRE[0], 0) + compte.get(ORDRE[1], 0))
print(f"{n:,} lignes analysées".replace(",", " "))
print(f"  sièges sociaux  : {100*n_siege/n:5.1f} % du total · {100*n_siege/n_res:5.1f} % des résolus")
print(f"  établissements  : {100*compte.get('établissement',0)/n:5.1f} % du total · "
      f"{100*compte.get('établissement',0)/n_res:5.1f} % des résolus")
resume

In [ ]:
import matplotlib.pyplot as plt

SURFACE, INK = "#fcfcfb", "#0b0b0b"
COULEURS = {"siège social (avec établissement)": "#2a78d6",
            "siège social (sans établissement)": "#eb6834",
            "établissement":                     "#1baf7a",
            "non résolu":                        "#a3a29a"}

libelles = ["Siège social\n(avec établissement)", "Siège social\n(sans établissement)",
            "Établissement", "Non résolu"]
vals = list(resume["n"])

fig, ax = plt.subplots(figsize=(8.5, 3.8), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
barres = ax.barh(libelles[::-1], vals[::-1], height=0.55,
                 color=[COULEURS[f] for f in ORDRE][::-1], edgecolor=SURFACE, linewidth=2)
for rect, v in zip(barres, vals[::-1]):
    ax.text(rect.get_width() + max(vals) * 0.015, rect.get_y() + rect.get_height() / 2,
            f"{v:,}".replace(",", " ") + f"   {100*v/n:.1f} %",
            va="center", ha="left", color=INK, fontsize=10)

ax.set_xlim(0, max(vals) * 1.3)
ax.set_title(f"Statut Sirene des {n:,} SIRET de visits_df".replace(",", " "),
             loc="left", color=INK, fontsize=12, pad=14)
ax.tick_params(axis="y", length=0, labelcolor=INK, labelsize=10)
ax.set_xticks([])
for s in ax.spines.values():
    s.set_visible(False)
fig.tight_layout()
fig.savefig("repartition_sieges.png", dpi=200, facecolor=SURFACE)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FLAG_CIBLE = "siège social (avec établissement)"
COL_NB     = "nb_etab_secondaires_actifs"

sieges = visits_df.loc[visits_df["flag"] == FLAG_CIBLE].copy()
sieges[COL_NB] = sieges[COL_NB].astype("Int64")

# une entreprise peut apparaître sur plusieurs lignes : on regarde les deux angles
par_entreprise = sieges.drop_duplicates(subset="siret_norm")

print(f"{len(sieges):,} lignes · {len(par_entreprise):,} sièges distincts".replace(",", " "))
print("\nNombre d'établissements secondaires actifs, par siège distinct :")
print(par_entreprise[COL_NB].describe(percentiles=[.25, .5, .75, .9, .99]).round(1).to_string())

BORNES = [0, 1, 2, 5, 10, 50, 200, np.inf]
ETIQ   = ["1", "2", "3–5", "6–10", "11–50", "51–200", "200+"]

def repartition(df, libelle):
    tranche = pd.cut(df[COL_NB].astype("float"), bins=BORNES, labels=ETIQ, right=True)
    t = tranche.value_counts().reindex(ETIQ).fillna(0).astype(int).to_frame(libelle)
    t[f"% {libelle}"] = (100 * t[libelle] / max(len(df), 1)).round(1)
    return t

repart = repartition(par_entreprise, "sièges").join(repartition(sieges, "lignes"))
repart.index.name = "nb établissements secondaires"
display(repart)

In [ ]:
SURFACE, INK, INK2, BLEU = "#fcfcfb", "#0b0b0b", "#52514e", "#2a78d6"

vals  = list(repart["sièges"])
total = sum(vals)

fig, ax = plt.subplots(figsize=(8.5, 3.6), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
barres = ax.bar(ETIQ, vals, width=0.66, color=BLEU, edgecolor=SURFACE, linewidth=2)

for rect, v in zip(barres, vals):
    if v:
        ax.text(rect.get_x() + rect.get_width() / 2, rect.get_height() + max(vals) * 0.02,
                f"{v:,}".replace(",", " ") + f"\n{100*v/total:.0f} %",
                ha="center", va="bottom", color=INK, fontsize=9, linespacing=1.35)

ax.set_ylim(0, max(vals) * 1.22)
ax.set_title(f"Taille des {total:,} sièges visités".replace(",", " "),
             loc="left", color=INK, fontsize=12, pad=34)
ax.text(0, 1.02, "nombre d'établissements secondaires actifs rattachés",
        transform=ax.transAxes, color=INK2, fontsize=9.5, va="bottom")
ax.tick_params(axis="x", length=0, labelcolor=INK, labelsize=10)
ax.set_yticks([])
for s in ax.spines.values():
    s.set_visible(False)
fig.tight_layout()
fig.savefig("repartition_taille_sieges.png", dpi=200, facecolor=SURFACE)
plt.show()

In [ ]:
import io, re, tarfile, time
from pathlib import Path
import pandas as pd, requests

ACCO_URL   = "https://echanges.dila.gouv.fr/OPENDATA/ACCO/"
ACCO_DIR   = Path("./data_acco")        # cache : 1 parquet par archive traitée
ACCO_TMP   = Path("./data_acco/tmp")    # archives en cours de téléchargement
MODE       = "disque"                   # "disque" = téléchargement résumable puis
                                        # parsing local (recommandé pour le stock
                                        # de 45 Go) ; "flux" = rien sur disque
EFFACER_APRES = True                    # supprimer le .tar.gz une fois parsé

ACCO_DIR.mkdir(parents=True, exist_ok=True)
ACCO_TMP.mkdir(parents=True, exist_ok=True)

RE_ARCHIVE = re.compile(r'href="((?:Freemium_acco_global|ACCO)_(\d{8}-\d{6})\.tar\.gz)"')

index = requests.get(ACCO_URL, timeout=120).text
trouvees = [{"nom": n, "ts": ts, "stock": n.startswith("Freemium")}
            for n, ts in RE_ARCHIVE.findall(index)]

stock = max((a for a in trouvees if a["stock"]), key=lambda a: a["ts"])
archives = [stock] + sorted((a for a in trouvees if not a["stock"] and a["ts"] > stock["ts"]),
                            key=lambda a: a["ts"])

print(f"Stock      : {stock['nom']}")
print(f"Incréments : {len(archives)-1} archives, de {archives[1]['ts']} à {archives[-1]['ts']}")

In [ ]:
import xml.etree.ElementTree as ET

CHAMPS = ["TITRE_TXT", "SIRET", "DATE_MAJ", "DATE_DEPOT", "DATE_EFFET",
          "DATE_FIN", "DATE_DIFFUSION", "CONFORME_VERSION_INTEGRALE"]
COLONNES = (["id"] + [c.lower() for c in CHAMPS] + ["themes", "signataires",
            "action", "archive_ts", "ordre"])
RE_ID = re.compile(r"(ACCO[A-Z]*[0-9]{6,})")

RE_SIRET = re.compile(rb"<[^<>]*SIRET[^<>]*>\s*([0-9]{9,14})")
RE_DATE  = re.compile(rb"<DATE_TEXTE>\s*([0-9]{4}-[0-9]{2}-[0-9]{2})")
RE_ID    = re.compile(r"(ACCO[A-Z]*[0-9]{6,})")
ENTETE   = 65536          # octets lus en tête de chaque XML (les métadonnées
                          # DILA sont en début de fichier)


def _flux_archive(a):
    """Renvoie un file-object sur le .tar.gz, en flux ou depuis le disque."""
    url = ACCO_URL + a["nom"]
    if MODE == "flux":
        r = requests.get(url, stream=True, timeout=(30, 600))
        r.raise_for_status()
        r.raw.decode_content = True
        return r.raw, None

    cible = ACCO_TMP / a["nom"]
    attendu = int(requests.head(url, timeout=60, allow_redirects=True)
                          .headers.get("Content-Length", 0))
    for tentative in range(6):
        deja = cible.stat().st_size if cible.exists() else 0
        if attendu and deja >= attendu:
            break
        entetes = {"Range": f"bytes={deja}-"} if deja else {}
        try:
            with requests.get(url, stream=True, timeout=(30, 600), headers=entetes) as r:
                r.raise_for_status()
                with open(cible, "ab" if deja else "wb") as f:
                    for chunk in r.iter_content(1 << 22):
                        f.write(chunk); deja += len(chunk)
                        print(f"\r  {a['nom']} {deja/1e9:6.2f} / {attendu/1e9:.2f} Go", end="")
            if not attendu or deja >= attendu:
                break
        except requests.RequestException as err:
            print(f"\n  coupure ({err}) — reprise dans 10 s")
            time.sleep(10)
    else:
        raise RuntimeError(f"téléchargement incomplet : {a['nom']}")
    print()
    return open(cible, "rb"), cible

def _txt(el):
    if el is None or el.text is None:
        return None
    t = el.text.strip()
    return t or None


def lire_meta(flux):
    """Renvoie le dict des métadonnées d'un XML d'accord, ou None."""
    try:
        for _, el in ET.iterparse(flux, events=("end",)):
            if el.tag != "META":
                continue
            acco = el.find("META_SPEC/META_ACCO")
            ident = _txt(el.find("META_COMMUN/ID"))
            if acco is None or ident is None:
                return None
            d = {"id": ident}
            for c in CHAMPS:
                d[c.lower()] = _txt(acco.find(c))
            d["themes"] = "|".join(
                t for t in (_txt(x) for x in acco.findall("THEMES/THEME/LIBELLE")) if t)
            d["signataires"] = "|".join(
                s for s in (_txt(x) for x in acco.findall("SIGNATAIRES/SIGNATAIRE")) if s)
            return d
    except ET.ParseError:
        return None
    return None


def parser_archive(a) -> Path:
    sortie = ACCO_DIR / f"{a['nom'].replace('.tar.gz', '')}.parquet"
    if sortie.exists():
        return sortie

    lignes, rejets, t0 = [], 0, time.time()
    flux, sur_disque = _flux_archive(a)
    try:
        with tarfile.open(fileobj=flux, mode="r|gz") as tar:
            for i, m in enumerate(tar):
                if not m.isfile():
                    continue
                nom = m.name.rsplit("/", 1)[-1]

                if "suppression" in nom.lower() and not nom.lower().endswith(".xml"):
                    contenu = tar.extractfile(m).read().decode("utf-8", "replace")
                    for ligne in contenu.splitlines():
                        ident = RE_ID.search(ligne)
                        if ident:
                            lignes.append({"id": ident.group(1), "action": "suppression"})
                    continue

                if not nom.lower().endswith(".xml"):
                    continue

                d = lire_meta(tar.extractfile(m))
                if d is None:
                    rejets += 1
                    continue
                d["action"] = "ajout"
                lignes.append(d)

                if i % 50000 == 0 and i:
                    print(f"\r  {a['nom']} : {i:,} fichiers · {time.time()-t0:.0f}s"
                          .replace(",", " "), end="")
    finally:
        flux.close()
        if sur_disque and EFFACER_APRES:
            sur_disque.unlink(missing_ok=True)

    df = pd.DataFrame(lignes).reindex(columns=COLONNES)
    df["conforme_version_integrale"] = df["conforme_version_integrale"].eq("true")
    df["archive_ts"] = a["ts"]
    df["ordre"] = range(len(df))
    df.to_parquet(sortie, index=False)
    print(f"\r  {a['nom']} : {len(df):,} entrées ({rejets:,} XML ignorés) "
          f"en {time.time()-t0:.0f}s".replace(",", " "))
    return sortie


fichiers = []
for a in archives:
    print(f"→ {a['nom']}")
    fichiers.append(parser_archive(a))

In [ ]:
SEULEMENT_EN_VIGUEUR = False      # True = exclut les accords dont DATE_FIN est passée

brut = pd.concat([pd.read_parquet(f) for f in fichiers], ignore_index=True)
brut = brut.sort_values(["archive_ts", "ordre"])

dernier = brut.drop_duplicates(subset="id", keep="last")
accords = dernier.loc[dernier["action"] == "ajout"].drop(columns=["action", "ordre"]).copy()

sans_siret = int(accords["siret"].isna().sum())
accords = accords.loc[accords["siret"].notna()].copy()
accords["siren"] = accords["siret"].str[:9]

if SEULEMENT_EN_VIGUEUR:
    fin = pd.to_datetime(accords["date_fin"], errors="coerce")
    accords = accords.loc[fin.isna() | (fin >= pd.Timestamp.today().normalize())]

accords.to_parquet("accords_acco.parquet", index=False)

print(f"{len(brut):,} entrées lues · {len(dernier):,} accords distincts".replace(",", " "))
print(f"{len(accords):,} accords retenus · {sans_siret:,} écartés faute de SIRET"
      .replace(",", " "))
print(f"{accords['siren'].nunique():,} entreprises couvertes".replace(",", " "))
accords[["id", "siret", "titre_txt", "date_depot", "date_fin", "themes"]].head()

In [ ]:
par_siret = accords.groupby("siret").size().rename("nb_accords_siret")
par_siren = accords.groupby("siren").size().rename("nb_accords_siren")

visits_df = visits_df.drop(columns=["nb_accords_siret", "nb_accords_siren"], errors="ignore")
visits_df["nb_accords_siret"] = visits_df["siret_siege"].map(par_siret)
visits_df["nb_accords_siren"] = visits_df["siret_siege"].str[:9].map(par_siren)

# siège identifié mais aucun accord déposé => 0, et non « inconnu »
connu = visits_df["siret_siege"].notna()
visits_df.loc[connu, ["nb_accords_siret", "nb_accords_siren"]] = (
    visits_df.loc[connu, ["nb_accords_siret", "nb_accords_siren"]].fillna(0)
)
visits_df[["nb_accords_siret", "nb_accords_siren"]] = (
    visits_df[["nb_accords_siret", "nb_accords_siren"]].astype("Int64")
)

sieges_uniq = visits_df.loc[connu].drop_duplicates(subset="siret_siege")
print(f"{len(sieges_uniq):,} sièges distincts".replace(",", " "))
print(f"  avec ≥1 accord sous leur propre SIRET : "
      f"{(sieges_uniq['nb_accords_siret'] > 0).mean()*100:.1f} %")
print(f"  avec ≥1 accord au niveau entreprise   : "
      f"{(sieges_uniq['nb_accords_siren'] > 0).mean()*100:.1f} %")
print("\nNombre d'accords par entreprise (sièges distincts) :")
print(sieges_uniq["nb_accords_siren"].describe(percentiles=[.5, .75, .9, .99]).round(1).to_string())

visits_df[[SIRET_COL, "flag", "siret_siege", "nb_accords_siret", "nb_accords_siren"]].head(10)